## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/PCA/animal

# where you will do the exercise
WORK_DIR=$HOME/pca_called_genotypes_animal

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.pca_called_animal_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf  $DATA/blue_wildebeest_thin* .
cp -r -sf $DATA/multiRunK7 .

echo --programs that are installed:--
which PCAone
which plink

echo; echo --- files in folder ---
ls

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.pca_called_animal_workdir"))[1]
setwd(work_d)
getwd()

# PCA of called genotypes: blue wildebeest

We use the same blue wildebeest data as in the ADMIXTURE analysis, as a plink
binary fileset of called genotypes. We run PCAone, compare with the admixture
proportions, then prune the data for linkage disequilibrium and see what
changes. Finally we build an identity-by-state tree.

If you have not done the [MDS and PCA by hand](pca_mds_and_svd.ipynb) exercise,
do that one first.

Let's perform PCA on the whole data (without LD pruning). We will use PCAone:

In [ ]:
PCAone -h

Shows the options. To perform the PCA use the following command

In [ ]:

PCAone -b blue_wildebeest_thin -o blue_wildebeest_thin

**Question**
 - How many SNPs and how many individuals did PCAone use?

 - look at the above output. How many SNPs and how many individuals?
 
 The default is to calculate the top 10 PCs. If you want more you can use the option --pc <INT> to choose a different number. However, let see what the top PCs capture. 
    
First let look the two first PCs as well as the admixture proportions estimated in the previous admixture exercises


In [ ]:
#read in code to plot admixture proportions ( plotAdmix function)
source("https://raw.githubusercontent.com/GenisGE/evalAdmix/master/visFuns.R")

options(repr.plot.width=12, repr.plot.height=12)
layout(matrix(c(1,1,2,3),nrow=2,by=T),height=c(2,4),width=2:1)

# Read in inferred admixture proportions
q <- read.table("multiRunK7/blue_wildebeest_noLD.7.Q_4")

#read in the population labels (first column of fam file)
tab <- table(pop <- read.table("blue_wildebeest_thin.fam")[,1])

#make the plot. 
plotAdmix(q,pop=pop,rotatelab=15,padj=0.15,cex.lab=1.4,col=c(3,5,8,4,2,6,7))


pca <- read.table("blue_wildebeest_thin.eigvecs")
#layout(matrix(1:2,nrow=1),w=c(4,2))
plot(pca[,1:2],col=as.integer(as.factor(pop))+1,ylab=paste("PC",1),xlab=paste("PC",2),cex.lab=1.5,cex=2,lwd=8)
plot.new()
legend("top",legend=names(tab),bty="n",xpd=T,cex=2,text.col=1:length(tab)+1,text.font=2)


**Questions**
 - What information do you get from the PCA that you don't get from the ADMIXTURE results?
 - Can you identify the admixed individuals?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pca_animal_structure.json")


 - What information do you get from the PCA that you don't get from the ADMIXTURE results?
 - Can you identify the admixed individuals?
 
 
 Lets see what the other PCs show. 
 

In [ ]:
options(repr.plot.width=12, repr.plot.height=16)

par(mfrow=c(3,2))
for(pc in 0:4)
    plot(pca[,pc*2+1:2],col=as.factor(pop),ylab=paste("PC",pc*2+2),xlab=paste("PC",pc*2+1),cex.lab=1.5,cex=2,lwd=8)


**Questions**
 - How many PCs are used to separate the populations?
 - What do you think is captured on PC 7 and 8, and on PC 9 and 10?

 - How many PCs are used to separate the populations?
 - What do you think is captured on PC 7 and 8?
 - What  do you think is captured on PC 9 and 10?
 

## Bonus exercise if there is time


Lets try to run the PCA after pruning LD (linkage disequillibrium) from the data. The noLD data was created in the admixture exercise. 
Run first the pruning with PCA  and then run PCA on the pruned data

In [ ]:
PCAone -b blue_wildebeest_thin -k 6 --ld -o pcaone  

**Question**
 - `--ld` makes PCAone compute LD **after** adjusting for population structure. Why would unadjusted LD be misleading in a structured sample?

In [ ]:
#see created files
ls pcaone*

Calculate LD between markers in a 1Mb sliding window and identify a set of SNPs so that the LD is less than $r^2<0.1$

In [ ]:
PCAone -B pcaone.residuals \
         --match-bim pcaone.mbim \
         --ld-r2 0.1 \
         --ld-bp 1000000 \
         -o pcaone
         

**Questions**
 - `--ld-r2 0.1` and `--ld-bp 1000000` set the pruning. What do the two numbers mean?
 - Why prune on LD at all before a PCA?

  The LD measure has been adjusted for population structure using the PCs. 
  - Why does the LD need to be adjusted for population structure?
  - Why do we use a sliding window?
  
 Lets make a new plink file containing only the markers in low LD

In [ ]:
echo --number of variants to be kept --
wc -l pcaone.ld.prune.in

echo -e "\n --Extract variants using plink --"
plink --bfile blue_wildebeest_thin \
    --extract pcaone.ld.prune.in  \
    --make-bed  \
    --out blue_wildebeest_noLD  \
    --chr-set 29


**Question**
 - How many variants were kept out of the original set?

We can now perform the PCA again on the pruned data

In [ ]:
PCAone  -b blue_wildebeest_noLD -o blue_wildebeest_noLD

**Question**
 - Compare the singular values before and after pruning. Which data set captures more of the population structure in its top PCs?


We can start by comparing the singular values. These are proportional to the variance explained so that higher values means that the corresponding PC captures more information about the data. 

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)

singularValues <- scan("blue_wildebeest_thin.eigvals")
singularValuesNoLD <- scan("blue_wildebeest_noLD.eigvals")

varianceExplained <- function(x)
    x^2/sum(x^2)

varExp <- varianceExplained(rbind(singularValues,singularValuesNoLD))


barplot(varExp*100,beside=T,col=2:3,legend=c("With LD","Without LD"),
       ylab="%variance explained",xlab="PC 1:10")

 - which data set captures the most information about the population structure from the first top PCs?
 
 Lets plot the PCs from the two data sets

In [ ]:
options(repr.plot.width=12, repr.plot.height=32)

tab <- table(pop <- read.table("blue_wildebeest_noLD.fam")[,1])


pcaNoLD <- read.table("blue_wildebeest_noLD.eigvecs")


par(mfrow=c(5,2))
for(pc in 0:4){
 
    plot(pca[,pc*2+1:2],col=as.factor(pop),ylab=paste("PC",pc*2+2),xlab=paste("PC",pc*2+1),cex.lab=1.5,cex=2,lwd=8,main="with LD")
    plot(pcaNoLD[,pc*2+1:2],col=as.factor(pop),ylab=paste("PC",pc*2+2),xlab=paste("PC",pc*2+1),cex.lab=1.5,cex=2,lwd=8,main="No LD")

    
    }

 - Which of the PCs from previous analysis capture LD and not population structure?
 - Do you think it is better to perform PCA with out without LD?

**Questions**
 - Which of the PCs from the unpruned analysis capture LD rather than population structure?
 - Do you think it is better to prune or not, and what do you lose by pruning?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pca_ld_pruning.json")


## Identity by state tree
As a last minute addition we can also make a neighbour joining tree by first computing identity-by-state distances between individuals which a just the proportion of sites between two individuals where they are different.
We can then load these distances into R and produce a NJ tree with the package APE.


In [ ]:
# get IBS distances with plink
plink --allow-extra-chr \
    --bfile blue_wildebeest_thin \
    --distance square 1-ibs \
    --chr-set 29 \
    --out plink


**Question**
 - The IBS tree is built from pairwise distances rather than from PCs. Does it group the individuals the same way?

In [ ]:
library(ape)

# read in distances
m <- as.matrix(read.table("plink.mdist", header = F))

# read id of individuals
id <- read.table("plink.mdist.id")
rownames(m) <- id$V2
colnames(m) <- id$V2

pops <- c(4,6,8,2,5,7,3)
names(pops) <- unique(id$V1)

plot(nj(m), tip.color = pops[id$V1], type = "unrooted", show.tip.label = TRUE)
add.scale.bar()
legend("topright",
       legend = names(pops),
       fill = pops,
          ncol=4)

 - Can you find the admixed individual?
 - Try removing the " type = 'unrooted'," argument from the plotting command above and see what happens to the tree

**Questions**
 - Can you find the admixed individual in the tree?
 - Try removing the outgroup and replotting. Does the structure become clearer?